# Pokémon — Partie 3 : entraînement

Régression linéaire, arbre de décision, forêt aléatoire et sauvegarde du modèle. Code basé sur le tutoriel du professeur.

In [ ]:
from pathlib import Path
import pandas as pnd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Exécuter le notebook depuis le dossier apprentissage/
assert Path("datas/dataset.csv").exists(), "Fichier manquant : apprentissage/datas/dataset.csv (exécuter depuis apprentissage/)"

## 1. Préparer les données

In [ ]:
# Chargement du dataset avec pour séparateur des tabulations
dataset = pnd.read_csv("datas/dataset.csv", delimiter="\t")
print("Colonnes du dataset :")
for indice, colonne in enumerate(dataset.columns):
    print(indice, colonne)

# Suppression des lignes ayant des valeurs manquantes
dataset = dataset.dropna(axis=0, how="any").copy()

# X : caractéristiques indices 5 à 11 inclus (7 valeurs)
X_colonnes = list(dataset.columns[5:12])
X = dataset.iloc[:, 5:12].values

# Le tutoriel donne deux indices contradictoires (16 et 17).
# On utilise le nom réel de la colonne pour éviter de choisir la mauvaise.
colonnes_cibles = [c for c in dataset.columns if "POURCENTAGE" in c.upper() and "VICTOIRE" in c.upper()]
if len(colonnes_cibles) != 1:
    raise ValueError(f"Colonne de pourcentage de victoires introuvable ou ambiguë : {colonnes_cibles}")
colonne_cible = colonnes_cibles[0]
y = dataset[colonne_cible].values

print("\nCaractéristiques :", X_colonnes)
print("Colonne cible :", colonne_cible)
print("Observations après nettoyage :", len(dataset))
assert X.shape[1] == 7

# Construction du jeu d'entraînement et du jeu de validation
X_APPRENTISSAGE, X_VALIDATION, Y_APPRENTISSAGE, Y_VALIDATION = train_test_split(
    X, y, test_size=0.2, random_state=0
)

## 2. Régression linéaire

In [ ]:
algorithme_lineaire = LinearRegression()
algorithme_lineaire.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme_lineaire.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)
print(">> ----------- REGRESSION LINEAIRE -----------")
print(">> R² validation =", precision)
print(">> R² apprentissage =", algorithme_lineaire.score(X_APPRENTISSAGE, Y_APPRENTISSAGE))

## 3. Arbre de décision

In [ ]:
algorithme_arbre = DecisionTreeRegressor(random_state=0)
algorithme_arbre.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme_arbre.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)
print(">> ----------- ARBRE DE DECISION -----------")
print(">> R² validation =", precision)
print(">> R² apprentissage =", algorithme_arbre.score(X_APPRENTISSAGE, Y_APPRENTISSAGE))

## 4. Forêt aléatoire et sauvegarde du modèle

In [ ]:
algorithme = RandomForestRegressor(random_state=0)
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)
print(">> ----------- FORET ALEATOIRE -----------")
print(">> R² validation =", precision)
print(">> R² apprentissage =", algorithme.score(X_APPRENTISSAGE, Y_APPRENTISSAGE))

Path("modele").mkdir(exist_ok=True)
fichier = "modele/modele_pokemon.mod"
joblib.dump(algorithme, fichier)
print("Modèle sauvegardé :", fichier)